# 06 · Subqueries & CTEs

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~30 min**

## Learning objectives
- use a subquery in `WHERE` and in `FROM`;
- name intermediate results with `WITH ... AS` (a CTE);
- rank rows within groups with a window function.

A subquery is a query inside another query. A CTE (`WITH`) is a named subquery that makes complex queries readable. We also take a quick look at window functions.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [1]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.6/397.6 kB 23.9 MB/s eta 0:00:00


Connecting to 'sqlite:///../data/omics.db'

Connected to omics.db — you are ready.


## 1. Scalar subquery in WHERE
Samples whose pH is above the study average; the inner query computes that average.

In [ ]:
%%sql
SELECT sample_id, ph
FROM samples
WHERE ph > (SELECT AVG(ph) FROM samples);

A scalar subquery can also sit on either side of a comparison. Here we compare each temperature to the study-wide average the other way round.

In [ ]:
%%sql
-- samples COLDER than average (scalar subquery on the other side of <)
SELECT sample_id, temperature_c
FROM samples
WHERE temperature_c < (SELECT AVG(temperature_c) FROM samples)
ORDER BY temperature_c;

## 2. Subquery with IN
Genera of the taxa that are abundant overall (> 2000 reads).

In [ ]:
%%sql
SELECT genus, phylum FROM taxa
WHERE taxon_id IN (
    SELECT taxon_id FROM abundance GROUP BY taxon_id HAVING SUM(count) > 2000
);

`NOT IN` with a subquery gives the complement: taxa that are never abundant.

In [ ]:
%%sql
-- the complement of the previous query: taxa that are NOT abundant overall
-- (every taxon whose total reads never exceed 2000)
SELECT genus, phylum FROM taxa
WHERE taxon_id NOT IN (
    SELECT taxon_id FROM abundance GROUP BY taxon_id HAVING SUM(count) > 2000
)
ORDER BY phylum, genus;

## 3. Subquery in FROM (a derived table)
First total reads per sample, then average those totals per environment.

In [ ]:
%%sql
SELECT environment, ROUND(AVG(total_reads)) AS mean_library_size
FROM (
    SELECT s.environment, s.sample_id, SUM(a.count) AS total_reads
    FROM abundance a JOIN samples s ON a.sample_id = s.sample_id
    GROUP BY s.sample_id
) AS per_sample
GROUP BY environment;

## 4. The same, but readable with a CTE
`WITH` defines a temporary named result you can then query, which reads more clearly than nesting.

In [ ]:
%%sql
WITH sample_reads AS (
    SELECT sample_id, SUM(count) AS total_reads
    FROM abundance GROUP BY sample_id
)
SELECT s.environment, ROUND(AVG(sr.total_reads)) AS mean_library_size
FROM sample_reads sr
JOIN samples s ON sr.sample_id = s.sample_id
GROUP BY s.environment;

A CTE is reusable: define library size once, then answer a different question from it, such as the richest samples.

In [ ]:
%%sql
-- top 5 samples by library size, built on the same CTE idea
WITH sample_reads AS (
    SELECT sample_id, SUM(count) AS total_reads
    FROM abundance GROUP BY sample_id
)
SELECT sr.sample_id, s.environment, sr.total_reads
FROM sample_reads sr
JOIN samples s ON sr.sample_id = s.sample_id
ORDER BY sr.total_reads DESC
LIMIT 5;

## 5. Bonus — a window function
`RANK() OVER (PARTITION BY … ORDER BY …)` ranks rows within each group without collapsing them. Here, the rank of each taxon within two samples.

In [ ]:
%%sql
SELECT sample_id, taxon_id, count,
       RANK() OVER (PARTITION BY sample_id ORDER BY count DESC) AS rank_in_sample
FROM abundance
WHERE sample_id IN ('S001', 'S002')
ORDER BY sample_id, rank_in_sample;

---
## Exercises

**Exercise.** Samples warmer than the average temperature (scalar subquery).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, temperature_c FROM samples
WHERE temperature_c > (SELECT AVG(temperature_c) FROM samples);
```
</details>

**Exercise.** List the phyla of taxa that appear in more than 25 samples (subquery with IN).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT DISTINCT phylum FROM taxa
WHERE taxon_id IN (
    SELECT taxon_id FROM abundance GROUP BY taxon_id HAVING COUNT(DISTINCT sample_id) > 25
);
```
</details>

**Exercise.** Using a CTE, give the mean library size per treatment.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
WITH sample_reads AS (
    SELECT sample_id, SUM(count) AS total_reads
    FROM abundance GROUP BY sample_id
)
SELECT s.treatment, ROUND(AVG(sr.total_reads)) AS mean_library_size
FROM sample_reads sr JOIN samples s ON sr.sample_id = s.sample_id
GROUP BY s.treatment;
```
</details>

### Recap
- A subquery is a query used inside `WHERE`, `FROM`, or a column.
- `WITH name AS (…)` (a CTE) names a step and keeps queries readable.
- Window functions (`RANK() OVER (PARTITION BY …)`) summarise without collapsing rows.
- Next: 07 · Complete & complex queries.